<a href="https://colab.research.google.com/github/Eshankulov/cs-31-24-Adenov/blob/main/class_work_04_11_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q xgboost lightgbm catboost

In [13]:
import os, numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

if os.path.exists("o_churn_synthetic.csv"):
    df = pd.read_csv("o_churn_synthetic.csv")
else:
    print("CSV не найден, генерирую синтетические данные")
    rng = np.random.default_rng(42)
    n = 3000
    df = pd.DataFrame({
        "subscriber_code": np.arange(n),
        "region": rng.choice(["Bishkek","Osh","Jalal-Abad","Karakol","Naryn","Talas","Batken"], n),
        "tariff": rng.choice(["start","basic","plus","max"], n),
        "tenure_months": rng.integers(1, 72, n),
        "monthly_fee": rng.normal(500, 150, n).round(1),
        "data_gb": rng.gamma(2, 3, n).round(2),
        "calls_min": rng.gamma(3, 100, n).round(0),
        "support_calls": rng.poisson(1.5, n),
        "late_payments": rng.poisson(0.5, n),
        "roaming": rng.integers(0, 2, n),
    })
    z = (-1.2 + 0.5*df.support_calls + 0.6*df.late_payments - 0.02*df.tenure_months
         + 0.002*(df.monthly_fee-500) + 0.4*(df.tariff=="start"))
    df["churn"] = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)

X = df.drop(columns=["churn", "subscriber_code"])
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

def make_prep():
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
        remainder="passthrough")

print("train:", X_train.shape, " test:", X_test.shape)

CSV не найден, генерирую синтетические данные
train: (2400, 9)  test: (600, 9)


In [14]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def fit_and_report(name, pipe):
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    sec = time.perf_counter() - t0
    f1_tr = f1_score(y_train, pipe.predict(X_train))
    f1_te = f1_score(y_test, pipe.predict(X_test))
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"{name:<14} F1 train={f1_tr:.3f}  F1 test={f1_te:.3f}  "
          f"AUC test={auc:.3f}  {sec:.2f} c")

In [15]:
from xgboost import XGBClassifier

xgb = make_pipeline(make_prep(), XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42))
fit_and_report("XGBoost", xgb)

XGBoost        F1 train=0.729  F1 test=0.438  AUC test=0.718  0.21 c


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(make_prep(), RandomForestClassifier(
    n_estimators=100, random_state=42))
fit_and_report("Случайный лес", rf)

Случайный лес  F1 train=1.000  F1 test=0.439  AUC test=0.704  0.83 c


In [17]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
p = make_prep().fit(X_tr)
Xtr, Xval = p.transform(X_tr), p.transform(X_val)
print("строк:", len(X_tr), len(X_val), len(X_test),
      " признаков после one-hot:", Xtr.shape[1])

строк: 1920 480 600  признаков после one-hot: 18


In [18]:
es = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4,
                   eval_metric="logloss", early_stopping_rounds=50,
                   random_state=42)
es.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
print("лучшая итерация:", es.best_iteration,
      " logloss на val:", round(es.best_score, 4))

лучшая итерация: 97  logloss на val: 0.5761


In [19]:
for lr in [0.3, 0.1, 0.05, 0.01]:
    for depth in [3, 4, 6]:
        m = XGBClassifier(n_estimators=3000, learning_rate=lr,
                          max_depth=depth, eval_metric="logloss",
                          early_stopping_rounds=50, random_state=42)
        m.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
        auc = roc_auc_score(y_val, m.predict_proba(Xval)[:, 1])
        n = m.best_iteration + 1
        print(f"lr={lr:<5} depth={depth}  деревьев={n:<4} AUC val={auc:.3f}")

lr=0.3   depth=3  деревьев=19   AUC val=0.701
lr=0.3   depth=4  деревьев=22   AUC val=0.720
lr=0.3   depth=6  деревьев=10   AUC val=0.685
lr=0.1   depth=3  деревьев=50   AUC val=0.705
lr=0.1   depth=4  деревьев=42   AUC val=0.710
lr=0.1   depth=6  деревьев=60   AUC val=0.698
lr=0.05  depth=3  деревьев=125  AUC val=0.706
lr=0.05  depth=4  деревьев=98   AUC val=0.708
lr=0.05  depth=6  деревьев=107  AUC val=0.695
lr=0.01  depth=3  деревьев=542  AUC val=0.707
lr=0.01  depth=4  деревьев=475  AUC val=0.708
lr=0.01  depth=6  деревьев=395  AUC val=0.698


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaled = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
    remainder=StandardScaler())

models = {
    "Логрегрессия": make_pipeline(scaled, LogisticRegression(max_iter=1000)),
    "Случайный лес": make_pipeline(make_prep(), RandomForestClassifier(
        n_estimators=100, random_state=42)),
    "XGBoost": make_pipeline(make_prep(), XGBClassifier(
        n_estimators=127, learning_rate=0.05, max_depth=3, random_state=42)),
}
for name, pipe in models.items():
    fit_and_report(name, pipe)

Логрегрессия   F1 train=0.483  F1 test=0.472  AUC test=0.734  0.08 c
Случайный лес  F1 train=1.000  F1 test=0.439  AUC test=0.704  1.15 c
XGBoost        F1 train=0.545  F1 test=0.449  AUC test=0.728  0.34 c


In [21]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, pipe in models.items():
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:<14} AUC CV = {s.mean():.3f} ± {s.std():.3f}")

Логрегрессия   AUC CV = 0.716 ± 0.035
Случайный лес  AUC CV = 0.678 ± 0.036
XGBoost        AUC CV = 0.692 ± 0.037


In [22]:
import pickle, statistics, warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb
from catboost import CatBoostClassifier

cat_cols = ["region", "tariff"]

# Категории: CatBoost берёт строки как есть; LightGBM — dtype category (нативно);
# XGBoost — one-hot через make_prep()
Xtr_c, Xte_c = X_train.copy(), X_test.copy()
for c in cat_cols:
    cats = pd.Categorical(X_train[c]).categories
    Xtr_c[c] = pd.Categorical(X_train[c], categories=cats)
    Xte_c[c] = pd.Categorical(X_test[c], categories=cats)

N_JOBS = 2
def build(name):
    if name == "XGBoost":
        return make_pipeline(make_prep(), XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            n_jobs=N_JOBS, random_state=42)), X_train, X_test
    if name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            n_jobs=N_JOBS, random_state=42, verbose=-1), Xtr_c, Xte_c
    return CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=4, thread_count=N_JOBS,
        random_seed=42, verbose=False, cat_features=cat_cols), X_train, X_test

rows = []
for name in ["XGBoost", "LightGBM", "CatBoost"]:
    fit_t, inf_t = [], []
    for i in range(5):                       # 5 замеров, первый выбрасываем
        model, Xa, Xb = build(name)
        t0 = time.perf_counter(); model.fit(Xa, y_train); fit_t.append(time.perf_counter()-t0)
        t0 = time.perf_counter(); proba = model.predict_proba(Xb)[:, 1]; inf_t.append(time.perf_counter()-t0)
    rows.append({
        "Библиотека": name,
        "AUC test": round(roc_auc_score(y_test, proba), 3),
        "Обучение, с": round(statistics.median(fit_t[1:]), 3),
        "Инференс, с": round(statistics.median(inf_t[1:]), 4),
        "Размер модели, КБ": round(len(pickle.dumps(model)) / 1024, 1),
    })
result = pd.DataFrame(rows)
result

,Библиотека,AUC test,"Обучение, с","Инференс, с","Размер модели, КБ"
0,XGBoost,0.718,1.432,0.0161,467.3
1,LightGBM,0.716,0.298,0.0185,410.0
2,CatBoost,0.734,1.059,0.0021,106.9


### Рекомендация (отредактируй по своим цифрам)
Для данных об оттоке такого размера (3000 строк, 2 категориальных признака) разница по AUC между тремя библиотеками, скорее всего, в пределах шума. Тогда решают другие критерии:

- **CatBoost** выбираем, если категорий станет много (десятки регионов/тарифов): нативная работа с категориями без one-hot и обычно устойчивее на малых данных.
- **LightGBM** выбираем, если данных станет сильно больше (сотни тысяч строк и выше) и важна скорость обучения.
- **XGBoost** оставляем, если нужна совместимость с уже существующим пайплайном/продакшеном и привычный one-hot.

Выбор изменился бы: при росте данных на порядки (→ LightGBM), при росте числа категорий (→ CatBoost), при жёстких требованиях к размеру модели или времени инференса (смотри таблицу выше).

*Честное время:* одинаковый `n_jobs`, 5 замеров, первый выброшен, берётся медиана.